<div align="center">

# AI-Powered Real-Time Noise Cancellation Adaptive System for Multiple Speaker

### Stage A: Google Colab audio classification pipeline

**Three classes | 16 kHz mono audio | 2 second windows | CNN on mel-spectrograms**

</div>

This notebook prepares the classifier that will later be integrated into the MATLAB real-time system. It is deliberately limited to **Stage A**. Adaptive noise cancellation, source separation, microphone streaming, and real-time output belong to Stage B and are not implemented here.

| Class ID | Meaning | Audio presented to the classifier |
| --- | --- | --- |
| `0` | Clean Speech | A standardized speech window |
| `1` | Environmental Noise | A standardized environmental-noise window |
| `2` | Noisy Speech | Speech mixed with controlled environmental noise |

**Run order:** execute the cells from top to bottom in a fresh Colab runtime. Begin with `QUICK_TEST_MODE = True` so the complete pipeline can be checked before committing to a long training run. Quick-test metrics are pipeline checks, not final FYP results.

## Technical implementation plan and risk audit

The pipeline splits speakers and complete noise recordings before segmentation, measures SNR after mixing, uses only verified RIR files, fits normalization on training data, and stores configuration/provenance with exported artifacts. These checks target leakage, clipping, invalid archives, normalization leakage, and Colab storage instability.

# Stage 1 - Environment setup

## Step 1 - Install and verify dependencies

Install the audio, data, plotting, machine-learning, and export packages before importing them. The import and version checks make runtime compatibility visible before any dataset work begins.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'librosa', 'soundfile', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'requests', 'tqdm'], check=False)

In [ ]:
import os
import gc
import csv
import json
import math
import random
import hashlib
import shutil
import tarfile
import zipfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.signal import fftconvolve
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
import tensorflow as tf
from IPython.display import Audio, display

## Step 2 - Check the installed versions

This check records Python, TensorFlow, and librosa versions before audio processing and model training.

In [ ]:
print('Python:', subprocess.check_output(['python', '--version'], text=True).strip())
print('TensorFlow:', tf.__version__)
print('librosa:', librosa.__version__)
print('PASS: Core imports succeeded.')

# Stage 1 - Environment setup

## Step 2 - Configure Colab/local storage and one reproducible experiment

The next cell chooses Colab or local paths, defines audio and model settings, creates persistent and temporary directories, fixes random seeds, and generates a configuration ID.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNNING_IN_COLAB = True
except ModuleNotFoundError:
    RUNNING_IN_COLAB = False
    print('Local runtime detected; Google Drive mounting is skipped.')

QUICK_TEST_MODE = True
SEED = 20260828
PROJECT_TITLE = 'AI-Powered Real-Time Noise Cancellation Adaptive System for Multiple Speaker'
SAMPLE_RATE = 16000
CHANNELS = 1
SEGMENT_SECONDS = 2.0
SAMPLES_PER_SEGMENT = int(SAMPLE_RATE * SEGMENT_SECONDS)
SNR_LEVELS_DB = [15, 10, 5, 0, -5, -10]
CLASS_NAMES = {0: 'Clean Speech', 1: 'Environmental Noise', 2: 'Noisy Speech'}
FEATURE_CONFIG = {'sample_rate': SAMPLE_RATE, 'n_fft': 512, 'hop_length': 256, 'win_length': 512, 'n_mels': 64, 'fmin': 20, 'fmax': 7600, 'power': 2.0, 'center': False, 'db_conversion': 'librosa.power_to_db(ref=np.max)'}
MAX_SPEECH_FILES = 80 if QUICK_TEST_MODE else 3000
MAX_NOISE_FILES = 60 if QUICK_TEST_MODE else 2000
SEGMENTS_PER_SPEECH_FILE = 2 if QUICK_TEST_MODE else 8
SEGMENTS_PER_NOISE_FILE = 2 if QUICK_TEST_MODE else 8
MAX_RECORDS_PER_CLASS_SPLIT = 60 if QUICK_TEST_MODE else 5000
EPOCHS = 2 if QUICK_TEST_MODE else 40
BATCH_SIZE = 8 if QUICK_TEST_MODE else 32
RIR_PROBABILITY = 0.25
DATASET_PLAN_VERSION = 'mini-librispeech-train-clean-5+esc50-master'
EXTERNAL_DATASET_ROOT = Path(os.environ.get('FYP_EXTERNAL_DATASET_ROOT', '/content/external_datasets'))

if RUNNING_IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive/FYP')
    LOCAL_ROOT = Path('/content/fyp_local')
else:
    DRIVE_ROOT = Path.cwd() / 'FYP_drive'
    LOCAL_ROOT = Path.cwd() / 'fyp_local'
RAW_ROOT = LOCAL_ROOT / 'raw'
CONFIG_DIR = DRIVE_ROOT / 'configuration'
METADATA_DIR = DRIVE_ROOT / 'metadata'
MANIFEST_DIR = DRIVE_ROOT / 'manifests'
MODEL_DIR = DRIVE_ROOT / 'models'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
RESULT_DIR = DRIVE_ROOT / 'results'
PLOT_DIR = DRIVE_ROOT / 'plots'
EXPORT_DIR = DRIVE_ROOT / 'exports'
MATLAB_DIR = DRIVE_ROOT / 'matlab_integration'
VERIFIED_RIR_DIR = LOCAL_ROOT / 'verified_rirs'
for directory in [DRIVE_ROOT, LOCAL_ROOT, RAW_ROOT, CONFIG_DIR, METADATA_DIR, MANIFEST_DIR, MODEL_DIR, CHECKPOINT_DIR, RESULT_DIR, PLOT_DIR, EXPORT_DIR, MATLAB_DIR, VERIFIED_RIR_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
CONFIG_ID = hashlib.sha256(json.dumps({'seed': SEED, 'quick_test': QUICK_TEST_MODE, 'dataset_plan': DATASET_PLAN_VERSION, 'feature': FEATURE_CONFIG}, sort_keys=True).encode()).hexdigest()[:12]
print('Configuration ID:', CONFIG_ID)
print('PASS: Configuration and storage directories are ready.')

# Stage 2 - Dataset acquisition

## Step 3 - Build a robust dataset portfolio

Mini LibriSpeech provides clean speech for the quick test and ESC-50 provides labeled environmental noise. Other sources such as LibriSpeech, MUSAN, DEMAND, DNS, WHAM, WHAMR, LibriMix, and SLR28 remain documented as optional or future resources rather than being silently merged into this experiment. Complete source recordings stay within one split.

In [ ]:
DATASET_SOURCES = {
    'Mini LibriSpeech': {'role': 'clean_speech', 'kind': 'speech', 'status': 'AUTO_DOWNLOAD', 'quick_download': True, 'url': 'https://www.openslr.org/resources/31/train-clean-5.tar.gz', 'archive_name': 'mini_librispeech.tar.gz', 'license': 'LibriSpeech terms require verification', 'expected_formats': ['flac'], 'local_path': RAW_ROOT / 'mini_librispeech', 'access': 'public URL'},
    'LibriSpeech train-clean-100': {'role': 'clean_speech', 'kind': 'speech', 'status': 'REQUIRES_DOWNLOAD', 'quick_download': False, 'url': 'https://www.openslr.org/12', 'archive_name': 'train-clean-100.tar.gz', 'license': 'LibriSpeech terms require verification', 'expected_formats': ['flac'], 'local_path': EXTERNAL_DATASET_ROOT / 'train-clean-100', 'access': 'manual download'},
    'MUSAN': {'role': 'noise', 'kind': 'noise', 'status': 'REQUIRES_DOWNLOAD', 'quick_download': False, 'url': 'https://www.openslr.org/17/', 'archive_name': 'musan.tar.gz', 'license': 'MUSAN terms require verification', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'musan', 'access': 'manual download'},
    'DEMAND': {'role': 'noise', 'kind': 'noise', 'status': 'REQUIRES_DOWNLOAD', 'quick_download': False, 'url': 'https://zenodo.org/record/1227121', 'archive_name': 'demand.zip', 'license': 'DEMAND terms require verification', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'demand', 'access': 'manual download'},
    'ESC-50': {'role': 'noise', 'kind': 'noise', 'status': 'AUTO_DOWNLOAD', 'quick_download': True, 'url': 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip', 'archive_name': 'esc50.zip', 'license': 'CC BY-NC; verify use', 'expected_formats': ['wav', 'csv'], 'local_path': RAW_ROOT / 'esc50', 'access': 'public URL'},
    'DNS': {'role': 'noisy_speech', 'kind': 'noisy_speech', 'status': 'REQUIRES_DOWNLOAD', 'quick_download': False, 'url': 'https://dnschallenge.microsoft.com/', 'archive_name': '', 'license': 'DNS access terms require verification', 'expected_formats': ['wav', 'json'], 'local_path': EXTERNAL_DATASET_ROOT / 'dns', 'access': 'restricted/manual'},
    'WHAM': {'role': 'noisy_speech', 'kind': 'noisy_speech', 'status': 'REQUIRES_DOWNLOAD', 'quick_download': False, 'url': 'https://wham.whisper.ai/', 'archive_name': '', 'license': 'WHAM terms require verification', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'wham', 'access': 'manual download'},
    'WHAMR': {'role': 'reverb_noisy', 'kind': 'noisy_speech', 'status': 'REQUIRES_DOWNLOAD', 'quick_download': False, 'url': 'https://wham.whisper.ai/', 'archive_name': '', 'license': 'WHAMR terms require verification', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'whamr', 'access': 'manual download'},
    'LibriMix': {'role': 'future_multi_speaker', 'kind': 'multi_speaker', 'status': 'REQUIRES_DOWNLOAD', 'quick_download': False, 'url': 'https://github.com/JorisCos/LibriMix', 'archive_name': '', 'license': 'LibriMix terms require verification', 'expected_formats': ['wav', 'json'], 'local_path': EXTERNAL_DATASET_ROOT / 'librimix', 'access': 'manual download'},
    'SLR28': {'role': 'rir', 'kind': 'rir', 'status': 'SKIPPED', 'quick_download': False, 'url': 'https://www.openslr.org/28/', 'archive_name': '', 'license': 'SLR28 terms require verification', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'slr28', 'access': 'manual download and verification required'}
}
DATASET_REGISTRY = list(DATASET_SOURCES)

def download_file(url, destination, retries=3):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 1024:
        return destination
    for attempt in range(1, retries + 1):
        temporary = destination.with_suffix(destination.suffix + '.part')
        try:
            with requests.get(url, stream=True, timeout=60, headers={'User-Agent': 'FYP-Colab-Pipeline/1.0'}) as response:
                response.raise_for_status()
                with temporary.open('wb') as output:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk: output.write(chunk)
            if temporary.stat().st_size <= 1024: raise RuntimeError('Downloaded archive is unexpectedly small.')
            temporary.replace(destination)
            return destination
        except Exception as error:
            if temporary.exists(): temporary.unlink()
            if attempt == retries: raise RuntimeError(f'Unable to download {url}: {error}') from error

def extract_archive(archive, extract_dir):
    extract_dir = Path(extract_dir)
    marker = extract_dir / '.extracted_ok'
    if marker.exists(): return extract_dir
    extract_dir.mkdir(parents=True, exist_ok=True)
    if str(archive).endswith(('.tar.gz', '.tgz')):
        with tarfile.open(archive, 'r:gz') as handle: handle.extractall(extract_dir)
    elif str(archive).endswith('.zip'):
        with zipfile.ZipFile(archive) as handle: handle.extractall(extract_dir)
    else: raise ValueError(f'Unsupported archive type: {archive}')
    marker.write_text('ok', encoding='utf-8')
    return extract_dir

def audio_file_count(root, formats):
    return sum(len(list(Path(root).rglob(f'*.{extension}'))) for extension in formats if Path(root).exists())

def scan_external_dataset_root(root=EXTERNAL_DATASET_ROOT):
    return {name: audio_file_count(source['local_path'], source['expected_formats']) for name, source in DATASET_SOURCES.items()}

def acquire_datasets():
    acquired = {}
    for name, source in DATASET_SOURCES.items():
        if not source['quick_download']: continue
        archive = RAW_ROOT / source['archive_name']
        if not Path(source['local_path']).exists() or audio_file_count(source['local_path'], source['expected_formats']) == 0:
            download_file(source['url'], archive)
            extract_archive(archive, source['local_path'])
        if audio_file_count(source['local_path'], source['expected_formats']) == 0: raise RuntimeError(f'No expected files found for {name}.')
        acquired[name] = source
    return acquired

ACQUIRED = acquire_datasets()
external_counts = scan_external_dataset_root()
dataset_inventory = pd.DataFrame([{'dataset': name, 'status': 'AVAILABLE' if audio_file_count(source['local_path'], source['expected_formats']) else ('REQUIRES_DOWNLOAD' if source['status'] == 'REQUIRES_DOWNLOAD' else 'SKIPPED'), 'role': source['role'], 'kind': source['kind'], 'file_count': audio_file_count(source['local_path'], source['expected_formats']), 'local_path': str(source['local_path']), 'source_url': source['url'], 'license_status': source['license'], 'access': source['access'], 'used_in_classifier': False} for name, source in DATASET_SOURCES.items()])
display(dataset_inventory)

# Stage 3 - Inspect audio and build source manifests

## Step 4 - Measure the datasets before using them

The manifest records paths, measured audio metadata, speaker identity, and noise labels. These source-level records are the basis for auditable splitting and later provenance.

In [ ]:
def audio_metadata(path):
    info = sf.info(str(path))
    return {'sample_rate': int(info.samplerate), 'channels': int(info.channels), 'duration': float(info.duration)}

def build_speech_manifest(root, limit, dataset_name='Mini LibriSpeech'):
    files = sorted(Path(root).rglob('*.flac')) + sorted(Path(root).rglob('*.wav'))
    rows = []
    for path in files[:limit]:
        metadata = audio_metadata(path)
        parts = path.parts
        speaker_id = next((parts[index + 1] for index, value in enumerate(parts[:-1]) if value in ('train-clean-5', 'train-clean-100', 'dev-clean')), parts[-3] if len(parts) >= 3 else 'unknown')
        rows.append({'dataset': dataset_name, 'speaker_id': str(speaker_id), 'filename': path.name, 'filepath': str(path.resolve()), **metadata})
    return pd.DataFrame(rows)

def find_esc_metadata(root):
    candidates = list(Path(root).rglob('esc50.csv'))
    if not candidates:
        return {}
    table = pd.read_csv(candidates[0])
    return {str(row.filename): {'noise_category': str(row.category), 'environment': str(row.category)} for _, row in table.iterrows()}

def build_noise_manifest(root, limit, dataset_name='ESC-50'):
    metadata_map = find_esc_metadata(root)
    files = sorted(Path(root).rglob('*.wav'))
    rows = []
    for path in files[:limit]:
        labels = metadata_map.get(path.name, {'noise_category': 'unknown', 'environment': 'unknown'})
        rows.append({'dataset': dataset_name, 'noise_source_id': str(path.resolve()), 'filename': path.name, 'filepath': str(path.resolve()), **labels, **audio_metadata(path)})
    return pd.DataFrame(rows)

speech_manifest = build_speech_manifest(ACQUIRED['Mini LibriSpeech']['local_path'], MAX_SPEECH_FILES)
noise_manifest = build_noise_manifest(ACQUIRED['ESC-50']['local_path'], MAX_NOISE_FILES)
speech_manifest.to_csv(METADATA_DIR / f'speech_manifest_{CONFIG_ID}.csv', index=False)
noise_manifest.to_csv(METADATA_DIR / f'noise_manifest_{CONFIG_ID}.csv', index=False)
print('Speech files:', len(speech_manifest), '| Noise files:', len(noise_manifest))
print('PASS: Source manifests contain the required columns.')

# Stage 4 - Leakage-safe splits and segmentation records

## Step 5 - Split independent sources before making windows

Speech speakers and complete noise recordings are assigned to exactly one split before two-second windows are created. This prevents neighboring segments from crossing train, validation, and test boundaries.

In [ ]:
def split_ids(ids, seed, train_fraction=0.70, validation_fraction=0.15):
    values = sorted(map(str, set(ids)))
    rng = np.random.default_rng(seed)
    rng.shuffle(values)
    if len(values) < 3:
        raise ValueError('At least three independent IDs are required for train/validation/test splitting.')
    train_end = max(1, int(len(values) * train_fraction))
    validation_end = min(len(values) - 1, train_end + max(1, int(len(values) * validation_fraction)))
    return {'train': set(values[:train_end]), 'validation': set(values[train_end:validation_end]), 'test': set(values[validation_end:])}

speech_split_ids = split_ids(speech_manifest.speaker_id, SEED)
noise_split_ids = split_ids(noise_manifest.noise_source_id, SEED + 1)
speech_manifest['split'] = speech_manifest.speaker_id.map(lambda value: next(name for name, values in speech_split_ids.items() if value in values))
noise_manifest['split'] = noise_manifest.noise_source_id.map(lambda value: next(name for name, values in noise_split_ids.items() if value in values))

def make_segments(manifest, source_id_column, segments_per_file, kind):
    rows = []
    for _, item in manifest.iterrows():
        max_start = max(0.0, item.duration - SEGMENT_SECONDS)
        starts = np.linspace(0, max_start, num=max(1, segments_per_file))
        for segment_index, start in enumerate(starts):
            row = item.to_dict()
            row.update({'source_id': str(item[source_id_column]), 'start_time': float(start), 'segment_duration': SEGMENT_SECONDS, 'segment_index': segment_index, 'kind': kind})
            rows.append(row)
    return pd.DataFrame(rows)

speech_segments = make_segments(speech_manifest, 'speaker_id', SEGMENTS_PER_SPEECH_FILE, 'speech')
noise_segments = make_segments(noise_manifest, 'noise_source_id', SEGMENTS_PER_NOISE_FILE, 'noise')
print(speech_segments.groupby('split').size().to_dict())
print(noise_segments.groupby('split').size().to_dict())
print('PASS: Segment records were generated after independent source splitting.')

In [ ]:
def load_standard_audio(path, start_time=0.0, duration=SEGMENT_SECONDS):
    audio, source_rate = librosa.load(
        str(path), sr=None, mono=True,
        offset=float(start_time), duration=float(duration)
    )
    audio = np.asarray(audio, dtype=np.float32)
    if source_rate != SAMPLE_RATE and len(audio):
        audio = librosa.resample(
            audio, orig_sr=source_rate, target_sr=SAMPLE_RATE
        ).astype(np.float32)
    if len(audio) < SAMPLES_PER_SEGMENT:
        audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
    return audio[:SAMPLES_PER_SEGMENT].astype(np.float32)

def rms(audio):
    audio = np.asarray(audio, dtype=np.float64)
    return float(np.sqrt(np.mean(np.square(audio))))

def mix_at_snr(speech, noise, target_snr_db):
    speech = np.asarray(speech, dtype=np.float32)
    noise = np.asarray(noise, dtype=np.float32)
    speech_level = rms(speech)
    noise_level = rms(noise)
    if speech_level < 1e-8:
        raise ValueError('Speech segment is silent; cannot define a stable SNR.')
    if noise_level < 1e-8:
        raise ValueError('Noise segment is silent; cannot define a stable SNR.')
    scaling_factor = speech_level / (
        noise_level * (10 ** (target_snr_db / 20.0))
    )
    scaled_noise = noise * scaling_factor
    mixture = speech + scaled_noise
    peak = max(float(np.max(np.abs(mixture))), 1e-8)
    if peak > 0.99:
        factor = 0.99 / peak
        speech = speech * factor
        scaled_noise = scaled_noise * factor
        mixture = mixture * factor
    measured = 20.0 * np.log10(rms(speech) / rms(scaled_noise))
    return mixture.astype(np.float32), float(measured)

def normalize_rir(rir):
    rir = np.asarray(rir, dtype=np.float32)
    rir = rir - np.mean(rir)
    energy = np.sqrt(np.sum(rir ** 2) + 1e-12)
    return rir / energy

def apply_rir(audio, rir):
    filtered = fftconvolve(
        audio, normalize_rir(rir), mode='full'
    )[:SAMPLES_PER_SEGMENT]
    peak = max(float(np.max(np.abs(filtered))), 1e-8)
    if peak > 1:
        return (filtered / peak * 0.95).astype(np.float32)
    return filtered.astype(np.float32)

def discover_verified_rirs():
    return sorted(
        path for path in VERIFIED_RIR_DIR.rglob('*.wav') if path.is_file()
    )

## Step 14 - Test the audio mathematics before creating examples

These small tests use synthetic signals so they do not depend on the downloaded datasets. The requested SNR values should be reproduced within the chosen tolerance, and RIR convolution should return finite audio with the expected length.

A passing SNR test means the noise scaling formula is behaving as intended. A passing RIR test means the convolution code is numerically usable; it does not claim that an external RIR is valid until that RIR has been inspected.

In [ ]:
verified_rirs = discover_verified_rirs()
test_speech = np.ones(SAMPLES_PER_SEGMENT, dtype=np.float32) * 0.1
test_noise = np.sin(np.linspace(0, 400 * np.pi, SAMPLES_PER_SEGMENT)).astype(np.float32) * 0.1
snr_checks = []
for target in SNR_LEVELS_DB:
    _, measured = mix_at_snr(test_speech, test_noise, target)
    snr_checks.append({
        'target_db': target,
        'measured_db': measured,
        'pass': abs(measured - target) < 0.05
    })
rir_unit = apply_rir(
    test_speech,
    np.exp(-np.arange(256, dtype=np.float32) / 30.0)
)
assert np.isfinite(rir_unit).all()
assert len(rir_unit) == SAMPLES_PER_SEGMENT

display(pd.DataFrame(snr_checks))
print(
    'PASS: SNR generation is within tolerance.'
    if all(item['pass'] for item in snr_checks)
    else 'FAIL: SNR generation check failed.'
)
print('PASS: RIR convolution unit test succeeded.')
print('Verified external RIR files:', len(verified_rirs))

# Stage 6 - Build a balanced classification manifest

## Step 7 - Construct the three classes deliberately

**Purpose:** create a balanced, reproducible experiment table without materializing every possible audio mixture on disk.

For each split, the notebook samples matching numbers of:

- clean speech windows;
- environmental-noise windows; and
- noisy-speech windows at the configured SNR levels.

Each noisy row keeps its speech source, noise source, category, target SNR, measured SNR, RIR path, and reverberation flag. This provenance makes later error analysis possible and prevents a high score from hiding a weak condition.

**Expected result:** the class-count table shows comparable counts for all three classes in each split, followed by a breakdown of noisy examples by SNR and reverberation. The configuration limits are intentionally small in quick-test mode.

In [ ]:
def rows_for_split(table, split):
    return table[table.split == split].reset_index(drop=True)

def build_classification_records():
    rng = np.random.default_rng(SEED)
    all_rows = []
    for split in ['train', 'validation', 'test']:
        speech_rows = rows_for_split(speech_segments, split)
        noise_rows = rows_for_split(noise_segments, split)
        if speech_rows.empty or noise_rows.empty:
            raise RuntimeError(f'No source records available for {split}.')
        count = min(
            len(speech_rows), len(noise_rows), MAX_RECORDS_PER_CLASS_SPLIT
        )
        speech_rows = speech_rows.iloc[
            rng.choice(len(speech_rows), count, replace=len(speech_rows) < count)
        ]
        noise_rows = noise_rows.iloc[
            rng.choice(len(noise_rows), count, replace=len(noise_rows) < count)
        ]
        for _, item in speech_rows.iterrows():
            all_rows.append({
                'split': split, 'class_id': 0,
                'class_name': CLASS_NAMES[0],
                'speech_filepath': item.filepath,
                'speech_source_id': item.source_id,
                'speech_start_time': item.start_time,
                'noise_filepath': '', 'noise_source_id': '',
                'noise_start_time': 0.0, 'noise_dataset': '',
                'noise_category': '', 'target_snr_db': np.nan,
                'measured_snr_db': np.nan, 'rir_filepath': '',
                'reverberant': False
            })
        for _, item in noise_rows.iterrows():
            all_rows.append({
                'split': split, 'class_id': 1,
                'class_name': CLASS_NAMES[1],
                'speech_filepath': '', 'speech_source_id': '',
                'speech_start_time': 0.0,
                'noise_filepath': item.filepath,
                'noise_source_id': item.source_id,
                'noise_start_time': item.start_time,
                'noise_dataset': item.dataset,
                'noise_category': item.noise_category,
                'target_snr_db': np.nan, 'measured_snr_db': np.nan,
                'rir_filepath': '', 'reverberant': False
            })
        for index in range(count):
            speech_item = speech_rows.iloc[index]
            noise_item = noise_rows.iloc[index]
            target = SNR_LEVELS_DB[index % len(SNR_LEVELS_DB)]
            use_rir = bool(verified_rirs) and rng.random() < RIR_PROBABILITY
            rir_path = str(verified_rirs[index % len(verified_rirs)]) if use_rir else ''
            speech_audio = load_standard_audio(
                speech_item.filepath, speech_item.start_time
            )
            noise_audio = load_standard_audio(
                noise_item.filepath, noise_item.start_time
            )
            if use_rir:
                speech_audio = apply_rir(
                    speech_audio, sf.read(rir_path, always_2d=False)[0]
                )
            _, measured = mix_at_snr(speech_audio, noise_audio, target)
            all_rows.append({
                'split': split, 'class_id': 2,
                'class_name': CLASS_NAMES[2],
                'speech_filepath': speech_item.filepath,
                'speech_source_id': speech_item.source_id,
                'speech_start_time': speech_item.start_time,
                'noise_filepath': noise_item.filepath,
                'noise_source_id': noise_item.source_id,
                'noise_start_time': noise_item.start_time,
                'noise_dataset': noise_item.dataset,
                'noise_category': noise_item.noise_category,
                'target_snr_db': target,
                'measured_snr_db': measured,
                'rir_filepath': rir_path,
                'reverberant': use_rir
            })
    return pd.DataFrame(all_rows)

## Step 19 - Generate the balanced classification manifest

This function pairs source records within each split and creates the three labels. Clean speech and environmental noise become classes 0 and 1. Class 2 is generated by mixing the selected speech and noise at one of the requested SNR levels.

Because examples are generated from paths and start times, the notebook keeps the provenance of every example without writing a new WAV file for every mixture.

In [ ]:
classification_manifest = build_classification_records()

source_paths = classification_manifest[['speech_filepath', 'noise_filepath']].replace('', np.nan).stack()
if not source_paths.map(Path).map(Path.exists).all():
    raise FileNotFoundError('A source file referenced by the classification manifest is missing.')

noisy_rows = classification_manifest[classification_manifest.class_id == 2]
snr_error = (noisy_rows.measured_snr_db - noisy_rows.target_snr_db).abs()
if not snr_error.empty and not (snr_error < 0.05).all():
    raise AssertionError(
        f'Generated noisy-speech SNR exceeded tolerance: max error={snr_error.max():.4f} dB'
    )

classification_manifest.to_csv(
    MANIFEST_DIR / f'classification_manifest_{CONFIG_ID}.csv',
    index=False
)
display(classification_manifest.groupby(['split', 'class_name']).size().unstack(fill_value=0))
display(
    classification_manifest[classification_manifest.class_id == 2]
    .groupby(['split', 'target_snr_db', 'reverberant'])
    .size()
    .reset_index(name='count')
)
print('PASS: Balanced classification manifest saved for config', CONFIG_ID)

# Stage 7 - Explicit leakage and data integrity checks

## Step 8 - Refuse to train on contaminated data

**Purpose:** turn the most important FYP risk, data leakage, into executable assertions.

**What is checked:**

- speaker IDs are disjoint across train, validation, and test;
- original noise-source IDs are disjoint across splits;
- referenced audio paths exist;
- generated source windows are not duplicated;
- every split contains all three classes;
- generated noisy examples remain close to their requested SNR.

**PASS** means the corresponding condition was actually tested and passed. **FAIL** raises an error or prints a failure message so the notebook cannot quietly continue with an invalid experiment.

In [ ]:
def check_disjoint_sets(split_map, label):
    split_names = list(split_map)
    for left_index, left_name in enumerate(split_names):
        for right_name in split_names[left_index + 1:]:
            overlap = split_map[left_name] & split_map[right_name]
            if overlap:
                raise AssertionError(f'FAIL: {label} leakage between {left_name} and {right_name}: {list(overlap)[:3]}')
    print(f'PASS: No {label} leakage detected.')

check_disjoint_sets(speech_split_ids, 'speaker')
check_disjoint_sets(noise_split_ids, 'noise-source')
for source_column in ['speech_source_id', 'noise_source_id']:
    source_rows = classification_manifest.loc[classification_manifest[source_column] != '', ['split', source_column]]
    source_split_counts = source_rows.groupby(source_column).split.nunique()
    assert source_split_counts.max() <= 1, f'FAIL: {source_column} appears in multiple dataset splits.'

window_key = ['split', 'speech_filepath', 'noise_filepath', 'speech_source_id', 'noise_source_id', 'speech_start_time', 'noise_start_time', 'class_id', 'target_snr_db']
duplicate_keys = classification_manifest.duplicated(subset=window_key, keep=False)
assert not duplicate_keys.any(), 'FAIL: Duplicate source windows detected.'
print('PASS: Source IDs and generated windows are isolated and non-duplicated.')
assert classification_manifest.groupby('split').class_id.nunique().eq(3).all()
print('PASS: Every split contains all three classes.')

# Stage 8 - Mel features and training-only normalization

## Step 9 - Convert audio into the CNN input

**Purpose:** represent each two-second waveform as a stable time-frequency image that a small CNN can classify.

The mel configuration is defined once near the beginning of the notebook. The feature function calculates the frame count from `n_fft`, `hop_length`, and the `center` policy; it does not assume that the result is always `64 x 63`.

**Normalization rule:** the mean and standard deviation are accumulated from training records only. Those same values are then applied to validation, test, and future inference audio. Validation or test data never contributes to the normalization statistics.

**Expected result:** the actual mel shape matches the calculated expected shape, all values are finite, and the normalization JSON records `training split only` as its source.

In [ ]:
def expected_feature_shape():
    if FEATURE_CONFIG['center']:
        frames = 1 + SAMPLES_PER_SEGMENT // FEATURE_CONFIG['hop_length']
    else:
        frames = 1 + (SAMPLES_PER_SEGMENT - FEATURE_CONFIG['n_fft']) // FEATURE_CONFIG['hop_length']
    return (FEATURE_CONFIG['n_mels'], frames)

EXPECTED_FEATURE_SHAPE = expected_feature_shape()

def audio_from_record(record):
    if int(record.class_id) == 0:
        return load_standard_audio(record.speech_filepath, record.speech_start_time)
    if int(record.class_id) == 1:
        return load_standard_audio(record.noise_filepath, record.noise_start_time)
    speech = load_standard_audio(record.speech_filepath, record.speech_start_time)
    noise = load_standard_audio(record.noise_filepath, record.noise_start_time)
    if bool(record.reverberant) and record.rir_filepath:
        speech = apply_rir(
            speech,
            sf.read(record.rir_filepath, always_2d=False)[0]
        )
    mixture, _ = mix_at_snr(speech, noise, float(record.target_snr_db))
    return mixture

def extract_mel_feature(audio):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=FEATURE_CONFIG['n_fft'],
        hop_length=FEATURE_CONFIG['hop_length'],
        win_length=FEATURE_CONFIG['win_length'],
        n_mels=FEATURE_CONFIG['n_mels'],
        fmin=FEATURE_CONFIG['fmin'],
        fmax=FEATURE_CONFIG['fmax'],
        power=FEATURE_CONFIG['power'],
        center=FEATURE_CONFIG['center']
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)

def feature_iterator(table):
    for _, record in table.iterrows():
        yield extract_mel_feature(audio_from_record(record)), int(record.class_id), record

## Step 16 - Check the mel-spectrogram dimensions

A mel-spectrogram is a compact picture of how energy changes across frequency and time. `n_mels` controls the number of frequency bands. `n_fft` controls the analysis-window size, and `hop_length` controls how far the window moves between frames.

Because `center=False` is used, the expected number of time frames is calculated from the 32,000-sample input rather than guessed. The next cell extracts one real feature and checks its shape and finite values.

In [ ]:
sample_feature, _, _ = next(
    feature_iterator(classification_manifest.head(1))
)
print('Expected feature shape:', EXPECTED_FEATURE_SHAPE)
print('Actual feature shape:', sample_feature.shape)
assert sample_feature.shape == EXPECTED_FEATURE_SHAPE
assert np.isfinite(sample_feature).all()
print('PASS: Feature shape and finite-value checks succeeded.')

## Step 17 - Calculate normalization from training data only

Neural networks usually train more steadily when input features are on a similar numerical scale. We calculate a mean and standard deviation for each mel location, but using validation or test data here would leak information from the evaluation set.

The following cell uses only `train_table`. The saved values are later reused unchanged for validation, test, and MATLAB inference.

In [ ]:
train_table = classification_manifest[
    classification_manifest.split == 'train'
].reset_index(drop=True)
feature_sum = np.zeros(EXPECTED_FEATURE_SHAPE, dtype=np.float64)
feature_square_sum = np.zeros(EXPECTED_FEATURE_SHAPE, dtype=np.float64)
feature_count = 0

for feature, _, _ in tqdm(
    feature_iterator(train_table),
    total=len(train_table),
    desc='Training normalization'
):
    feature_sum += feature
    feature_square_sum += feature.astype(np.float64) ** 2
    feature_count += 1

normalization_mean = feature_sum / max(feature_count, 1)
normalization_std = np.sqrt(
    np.maximum(
        feature_square_sum / max(feature_count, 1) - normalization_mean ** 2,
        1e-6
    )
).astype(np.float32)
normalization = {
    'mean': normalization_mean.tolist(),
    'std': normalization_std.tolist(),
    'source': 'training split only',
    'config_id': CONFIG_ID
}
(CONFIG_DIR / f'normalization_{CONFIG_ID}.json').write_text(
    json.dumps(normalization),
    encoding='utf-8'
)
print('Training records used:', feature_count)
print('PASS: Normalization statistics calculated from training data only.')

# Stage 9 - Lazy TensorFlow datasets and CNN

## Step 10 - Train without loading the corpus into RAM

**Purpose:** feed one standardized feature at a time to TensorFlow so Colab memory is not consumed by the entire raw dataset.

**What the next code cell does:** creates lazy `tf.data` generators, batches the records, defines a compact CNN, and checks the input/output dimensions. The model receives one mel image with a channel dimension and returns three softmax probabilities.

**Expected result:** `model.summary()` shows the convolutional network and the cell prints `PASS: Model input and output shapes are correct.`

In [ ]:
def normalized_feature(record):
    feature = extract_mel_feature(audio_from_record(record))
    return (
        (feature - normalization_mean) / normalization_std
    ).astype(np.float32)[..., np.newaxis]

def tensorflow_generator(table):
    for _, record in table.iterrows():
        yield normalized_feature(record), np.int32(record.class_id)

def make_tf_dataset(table, shuffle=False):
    dataset = tf.data.Dataset.from_generator(
        lambda: tensorflow_generator(table),
        output_signature=(
            tf.TensorSpec(
                shape=(*EXPECTED_FEATURE_SHAPE, 1),
                dtype=tf.float32
            ),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        dataset = dataset.shuffle(
            min(len(table), 1000),
            seed=SEED,
            reshuffle_each_iteration=False
        )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## Step 22 - Create lazy TensorFlow datasets

The generator loads one record, standardizes it, extracts its mel feature, and then yields it to TensorFlow. This is slower than loading everything into one large array, but it keeps memory use predictable for Colab.

The `shuffle` option is used for training only. Validation and test records keep their fixed order so predictions can be matched back to the manifest.

In [ ]:
validation_table = classification_manifest[
    classification_manifest.split == 'validation'
].reset_index(drop=True)
test_table = classification_manifest[
    classification_manifest.split == 'test'
].reset_index(drop=True)
train_dataset = make_tf_dataset(train_table, shuffle=True)
validation_dataset = make_tf_dataset(validation_table)
test_dataset = make_tf_dataset(test_table)
print('Training records:', len(train_table))
print('Validation records:', len(validation_table))
print('Testing records:', len(test_table))
print('PASS: Lazy TensorFlow datasets are ready.')

## Step 23 - Understand the CNN classifier

The model follows this path:

```text
Audio waveform
    ↓
Mel-spectrogram
    ↓
CNN filters learn time-frequency patterns
    ↓
Dense decision layer
    ↓
Three class probabilities
```

A convolution layer learns small local patterns. ReLU keeps useful positive responses, pooling reduces the feature-map size, batch normalization helps keep activations stable, and dropout randomly removes some connections during training to reduce overfitting. The final softmax layer produces one probability for each class.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*EXPECTED_FEATURE_SHAPE, 1)),
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()
assert model.input_shape[1:] == (*EXPECTED_FEATURE_SHAPE, 1)
assert model.output_shape[-1] == 3
print('PASS: Model input and output shapes are correct.')

# Stage 10 - Train and save the best validation model

## Step 11 - Select the checkpoint by validation performance

**Purpose:** train the classifier while reducing overfitting and keep the best validation checkpoint rather than assuming the final epoch is best.

**What the next code cell does:** fits the CNN, reduces the learning rate when validation loss stops improving, stops early when appropriate, restores the best weights, and saves the training history and accuracy/loss plot.

**Expected result:** training and validation curves are displayed, and a `.keras` checkpoint is saved. In quick-test mode this is only a pipeline verification; use full mode for meaningful FYP results.

In [ ]:
best_model_path = CHECKPOINT_DIR / f'best_classifier_{CONFIG_ID}.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(best_model_path),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5 if not QUICK_TEST_MODE else 1,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]
print('Best checkpoint:', best_model_path)
print('PASS: Training callbacks are configured.')

## Step 25 - Train the CNN

An epoch is one pass through the training records. A batch is the smaller group processed before the network updates its weights. The loss measures prediction error, while accuracy measures the fraction of correct labels.

Adam is the optimizer that updates the weights using the learning rate. Early stopping watches validation loss, and model checkpointing keeps the best validation model. The validation set guides model selection but is never used to calculate the test score.

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)
if best_model_path.exists():
    model = tf.keras.models.load_model(best_model_path)

history_json = {
    key: [float(value) for value in values]
    for key, values in history.history.items()
}
(RESULT_DIR / f'training_history_{CONFIG_ID}.json').write_text(
    json.dumps(history_json, indent=2),
    encoding='utf-8'
)
print('PASS: Training finished and best weights were restored when available.')

## Step 26 - Plot the learning curves

Training accuracy and validation accuracy show whether the model is learning patterns that also generalize beyond its training records. If training accuracy keeps rising while validation accuracy stops improving, the model may be overfitting.

The loss curves show the same relationship using prediction error. These plots are saved with the experiment configuration ID so they can be cited alongside the measured results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / f'training_curves_{CONFIG_ID}.png', dpi=150)
plt.show()
print('PASS: Training curves were saved.')

# Stage 11 - Untouched test evaluation and robustness analysis

## Step 12 - Measure performance only after model selection

**Purpose:** evaluate generalization on speakers and noise recordings that were kept completely outside training.

The test report includes overall accuracy, precision, recall, F1-score, a confusion matrix, saved predictions, and grouped results by noise dataset, noise category, and reverberation status. No test result is used to choose model weights or normalization statistics.

The noisy-speech subset is also evaluated separately at `+15`, `+10`, `+5`, `0`, `-5`, and `-10 dB`. This answers the engineering question that overall accuracy alone cannot: how does classification behave as the acoustic condition becomes harder?

**Expected result:** tables and plots are generated from real predictions. A quick-test score is a development check, not a claim about final system performance.

In [ ]:
def predictions_for_table(table):
    probabilities = model.predict(make_tf_dataset(table), verbose=0)
    return probabilities, np.argmax(probabilities, axis=1)

test_probabilities, test_predictions = predictions_for_table(test_table)
test_actual = test_table.class_id.to_numpy()
test_accuracy = float(accuracy_score(test_actual, test_predictions))
report = classification_report(test_actual, test_predictions, labels=[0, 1, 2], target_names=[CLASS_NAMES[i] for i in range(3)], output_dict=True, zero_division=0)
report_table = pd.DataFrame(report).T
display(report_table)
cm = confusion_matrix(test_actual, test_predictions, labels=[0, 1, 2])
plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt='d', cmap='crest', xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values()); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout(); plt.savefig(PLOT_DIR / f'confusion_matrix_{CONFIG_ID}.png', dpi=150); plt.show()
predictions_table = test_table.copy()
predictions_table['predicted_class_id'] = test_predictions
predictions_table['predicted_class'] = [CLASS_NAMES[int(value)] for value in test_predictions]
predictions_table['confidence'] = test_probabilities.max(axis=1)
predictions_table.to_csv(RESULT_DIR / f'test_predictions_{CONFIG_ID}.csv', index=False)

def grouped_accuracy(table, group_columns):
    rows = []
    for group_values, group in table.groupby(group_columns, dropna=False):
        if not isinstance(group_values, tuple): group_values = (group_values,)
        row = dict(zip(group_columns, group_values))
        row.update({'count': len(group), 'accuracy': accuracy_score(group.class_id, group.predicted_class_id)})
        rows.append(row)
    return pd.DataFrame(rows)

noise_dataset_results = grouped_accuracy(predictions_table[predictions_table.class_id == 2], ['noise_dataset'])
noise_category_results = grouped_accuracy(predictions_table[predictions_table.class_id == 2], ['noise_category'])
reverberation_results = grouped_accuracy(predictions_table[predictions_table.class_id == 2], ['reverberant'])
display(noise_dataset_results)
display(noise_category_results)
display(reverberation_results)
noise_dataset_results.to_csv(RESULT_DIR / f'test_by_noise_dataset_{CONFIG_ID}.csv', index=False)
noise_category_results.to_csv(RESULT_DIR / f'test_by_noise_category_{CONFIG_ID}.csv', index=False)
reverberation_results.to_csv(RESULT_DIR / f'test_by_reverberation_{CONFIG_ID}.csv', index=False)

robustness_rows = []
for snr in SNR_LEVELS_DB:
    subset = test_table[(test_table.class_id == 2) & (test_table.target_snr_db == snr)].reset_index(drop=True)
    if subset.empty: continue
    probabilities, predictions = predictions_for_table(subset)
    actual = subset.class_id.to_numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(actual, predictions, labels=[2], average='macro', zero_division=0)
    robustness_rows.append({'snr_db': snr, 'accuracy': accuracy_score(actual, predictions), 'precision': precision, 'recall': recall, 'f1': f1, 'count': len(subset)})
robustness_table = pd.DataFrame(robustness_rows)
display(robustness_table)
if not robustness_table.empty:
    plt.figure(figsize=(7, 4)); plt.plot(robustness_table.snr_db, robustness_table.accuracy, marker='o'); plt.gca().invert_xaxis(); plt.xlabel('Target SNR (dB)'); plt.ylabel('Noisy-speech accuracy'); plt.title('Noisy-Speech Accuracy vs SNR'); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(PLOT_DIR / f'snr_accuracy_{CONFIG_ID}.png', dpi=150); plt.show()
robustness_table.to_csv(RESULT_DIR / f'snr_robustness_{CONFIG_ID}.csv', index=False)
print('Test accuracy:', test_accuracy)
print('PASS: Test evaluation and SNR robustness results were generated.')

## Step 28 - Interpret the evaluation output

The classification report contains four useful views of performance:

- **Precision:** when the model predicts a class, how often that prediction is correct.
- **Recall:** how much of the actual class the model successfully finds.
- **F1-score:** a combined score that is high only when precision and recall are both useful.
- **Confusion matrix:** rows are the actual classes and columns are the predicted classes. The diagonal contains correct predictions; off-diagonal cells show the confusions.

The SNR table should be read separately from overall accuracy. Positive SNR means speech is stronger than noise, `0 dB` means their levels are approximately equal, and negative SNR means noise is stronger. The notebook reports the actual values from this run; it does not guess the cause of an error before seeing the results.

# Stage 12 - Listening and prediction demonstrations

## Step 13 - Listen to representative examples and inspect mistakes

**Purpose:** connect the tables to the audio and make the classifier behaviour easier to understand.

The next code cell provides clearly labelled players for clean speech, environmental noise, non-reverberant noisy speech, reverberant speech, and reverberant noisy speech when those examples exist. It normalizes audio only for comfortable playback; the model input is unchanged.

It also displays unseen test examples with their actual class, predicted class, and confidence. Incorrect predictions are grouped by class, SNR, and reverberation status. These groupings describe patterns in this run; they do not prove a cause without further experiments.

In [ ]:
from IPython.display import Audio, display

def normalize_for_playback(audio):
    peak = max(float(np.max(np.abs(audio))), 1e-8)
    return (audio / peak * 0.95).astype(np.float32)

demo_requests = [
    ('Clean speech', classification_manifest[classification_manifest.class_id == 0]),
    ('Environmental noise', classification_manifest[classification_manifest.class_id == 1]),
    ('Non-reverberant noisy speech', classification_manifest[(classification_manifest.class_id == 2) & (~classification_manifest.reverberant)]),
    ('Reverberant + noisy speech', classification_manifest[(classification_manifest.class_id == 2) & (classification_manifest.reverberant)]),
]
for label, candidates in demo_requests:
    if candidates.empty:
        print(f'Demo unavailable in this run: {label}')
        continue
    record = candidates.iloc[0]
    audio = normalize_for_playback(audio_from_record(record))
    print(f'Demo: {label}')
    display(Audio(audio, rate=SAMPLE_RATE))

if verified_rirs:
    reverberant_speech = apply_rir(
        audio_from_record(classification_manifest[classification_manifest.class_id == 0].iloc[0]),
        sf.read(verified_rirs[0], always_2d=False)[0]
    )
    print('Demo: Reverberant speech')
    display(Audio(normalize_for_playback(reverberant_speech), rate=SAMPLE_RATE))
else:
    print('Demo unavailable in this run: Reverberant speech (no verified RIR was supplied).')

display(predictions_table[['class_name', 'predicted_class', 'confidence', 'target_snr_db', 'reverberant']].head(10))
incorrect = predictions_table[predictions_table.class_id != predictions_table.predicted_class_id].copy()
print('Incorrect test examples:', len(incorrect))
if len(incorrect):
    display(incorrect.groupby(['class_name', 'predicted_class']).size().reset_index(name='count').sort_values('count', ascending=False))
    display(incorrect.groupby(['target_snr_db', 'reverberant']).size().reset_index(name='count'))
else:
    print('No incorrect examples in this runtime test set; no failure cause is inferred.')

# Stage 13 - Export for MATLAB R2022 integration

## Step 14 - Save the classifier and its preprocessing contract

**Purpose:** prevent MATLAB from guessing how the classifier input was made.

The JSON contract records the class IDs/names, sample rate, mono requirement, two-second window rule, mel parameters, dB conversion, normalization mean/std, expected dimensions, data type, and padding/truncation policy. The trained Keras model is saved as the authoritative model artifact.

ONNX export is attempted as an optional convenience. Compatibility depends on the MATLAB R2022 release and installed Deep Learning Toolbox, so an ONNX failure is reported without deleting or replacing the Keras model or JSON contract.

**Expected result:** the model, class mapping, normalization information, and versioned MATLAB configuration are saved under the configured persistent artifact folders.

In [ ]:
final_model_path = MODEL_DIR / f'noise_classifier_{CONFIG_ID}.keras'
model.save(final_model_path)
matlab_config = {
    'project_title': PROJECT_TITLE, 'config_id': CONFIG_ID, 'class_ids': CLASS_NAMES,
    'sample_rate': SAMPLE_RATE, 'channels': CHANNELS, 'audio_format': 'mono float32',
    'segment_duration_seconds': SEGMENT_SECONDS, 'samples_per_segment': SAMPLES_PER_SEGMENT,
    'padding_truncation': 'zero-pad at end; truncate at SAMPLES_PER_SEGMENT',
    'n_fft': FEATURE_CONFIG['n_fft'], 'hop_length': FEATURE_CONFIG['hop_length'], 'win_length': FEATURE_CONFIG['win_length'],
    'n_mels': FEATURE_CONFIG['n_mels'], 'fmin': FEATURE_CONFIG['fmin'], 'fmax': FEATURE_CONFIG['fmax'],
    'power': FEATURE_CONFIG['power'], 'center': FEATURE_CONFIG['center'], 'db_conversion': FEATURE_CONFIG['db_conversion'],
    'expected_mel_dimensions': list(EXPECTED_FEATURE_SHAPE), 'expected_cnn_input_dimensions': list((*EXPECTED_FEATURE_SHAPE, 1)),
    'model_input_dtype': 'float32', 'normalization_mean': normalization_mean.tolist(), 'normalization_std': normalization_std.tolist(),
    'model_file': str(final_model_path), 'normalization_source': 'training split only'
}
matlab_config_path = MATLAB_DIR / f'matlab_integration_config_{CONFIG_ID}.json'
matlab_config_path.write_text(json.dumps(matlab_config, indent=2), encoding='utf-8')
(EXPORT_DIR / f'class_mapping_{CONFIG_ID}.json').write_text(json.dumps(CLASS_NAMES, indent=2), encoding='utf-8')
print('PASS: Keras model saved:', final_model_path.exists())
print('PASS: MATLAB configuration saved:', matlab_config_path.exists())

try:
    import tf2onnx
    onnx_path = EXPORT_DIR / f'noise_classifier_{CONFIG_ID}.onnx'
    input_signature = [tf.TensorSpec((None, *EXPECTED_FEATURE_SHAPE, 1), tf.float32, name='mel_input')]
    onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature, opset=13)
    onnx_path.write_bytes(onnx_model.SerializeToString())
    print('OPTIONAL ONNX EXPORT PASS:', onnx_path)
except Exception as error:
    print('OPTIONAL ONNX EXPORT SKIPPED/FAILED:', error)
    print('The Keras model and MATLAB preprocessing contract remain the authoritative artifacts.')

# Stage 14 - Final automatic sanity check and project summary

## Step 15 - Report what was genuinely verified

**Purpose:** provide one final, auditable status report before the artifacts are used by MATLAB.

The final cell prints PASS or FAIL only for checks that were executed in this run. It also prints the dataset choices, split sizes, speaker count, SNR range, feature/CNN dimensions, measured validation/test metrics, model path, and MATLAB configuration path.

**Interpretation:** all required checks must pass before using the exported artifacts. A quick-test run confirms pipeline integrity but must not be presented as final FYP performance. A full run should be saved with its configuration ID and result files.

In [ ]:
used_speech = sorted(set(classification_manifest.loc[classification_manifest.speech_filepath != '', 'dataset']))
used_noise = sorted(set(classification_manifest.loc[classification_manifest.noise_dataset != '', 'noise_dataset']))
used_datasets = used_speech + used_noise
for dataset_name in used_datasets:
    dataset_inventory.loc[dataset_inventory.dataset == dataset_name, 'used_in_classifier'] = True
checks = {
    'Dataset inventory complete': set(DATASET_REGISTRY).issubset(set(dataset_inventory.dataset)),
    'Dataset acquisition': all(Path(item['local_path']).exists() for item in ACQUIRED.values()),
    'Audio standardization': len(audio_from_record(classification_manifest.iloc[0])) == SAMPLES_PER_SEGMENT,
    'Speaker leakage': all(not (speech_split_ids[a] & speech_split_ids[b]) for a in speech_split_ids for b in speech_split_ids if a < b),
    'Noise leakage': all(not (noise_split_ids[a] & noise_split_ids[b]) for a in noise_split_ids for b in noise_split_ids if a < b),
    'SNR generation': all(item['pass'] for item in snr_checks),
    'RIR processing': np.isfinite(rir_unit).all() and len(rir_unit) == SAMPLES_PER_SEGMENT,
    'Feature shape': sample_feature.shape == EXPECTED_FEATURE_SHAPE,
    'Normalization': normalization['source'] == 'training split only',
    'Model input': model.input_shape[1:] == (*EXPECTED_FEATURE_SHAPE, 1),
    'Model saved': final_model_path.exists(),
    'MATLAB configuration saved': matlab_config_path.exists()
}
for name, passed in checks.items(): print(f'{name:.<35} ' + ('PASS' if passed else 'FAIL'))
print('\nPROJECT SUMMARY')
print('Project title:', PROJECT_TITLE)
print('Configuration:', CONFIG_ID, '| Quick test mode:', QUICK_TEST_MODE)
print('Automatically acquired:', ', '.join(ACQUIRED) or 'None')
print('Classifier sources actually used:', ', '.join(used_datasets) or 'None')
print('Pending/restricted:', ', '.join(dataset_inventory.loc[dataset_inventory.status != 'AVAILABLE', 'dataset']) or 'None')
print('Samples train/validation/test:', *(int((classification_manifest.split == split).sum()) for split in ['train', 'validation', 'test']))
print('Speakers:', speech_manifest.speaker_id.nunique(), '| Classes:', list(CLASS_NAMES.values()))
print('SNR range (dB):', min(SNR_LEVELS_DB), 'to', max(SNR_LEVELS_DB), '| Sample rate:', SAMPLE_RATE, '| Segment seconds:', SEGMENT_SECONDS)
print('Mel shape:', EXPECTED_FEATURE_SHAPE, '| CNN input:', (*EXPECTED_FEATURE_SHAPE, 1))
print('Best validation accuracy:', max(history.history.get('val_accuracy', [float('nan')])), '| Final test accuracy:', test_accuracy)
print('Model location:', final_model_path)
print('MATLAB configuration:', matlab_config_path)
display(dataset_inventory[['dataset', 'status', 'role', 'kind', 'file_count', 'local_path', 'source_url', 'license_status', 'access', 'used_in_classifier']])
if not all(checks.values()):
    raise RuntimeError('One or more required sanity checks failed; inspect the failure above.')